# Trillion-Parameter Illusion: The CPU MoE Lab

This notebook accompanies the fact-checked article. No frontier checkpoint or GPU required. Start in the repo root.

## 1. See how routing selects experts

This uses random expert weights to isolate the mechanism. It is **not** a trained language model.

In [ ]:
import numpy as np
from moe_lab.core import ToyMoE
model = ToyMoE(n_experts=8, hidden=16, intermediate=32, seed=7)
x = np.random.default_rng(8).normal(size=(4, 16)).astype(np.float32)
ids, probabilities = model.routes(x, top_k=2)
print("selected experts:", ids)
print("routing weights:", probabilities)


## 2. Stored versus active weights

Top-k changes which feed-forward expert weights run, not which weights are stored.

In [ ]:
for k in [1, 2, 4, 8]:
    c = model.cost(k)
    print(f"k={k}: stored={c.total_parameters:,} active={c.active_parameters_per_token:,} MACs/token≈{c.estimated_macs_per_token:,} stored bytes={c.weight_bytes_float32:,}")


## 3. Check a correctness invariant

When every expert is selected and gate scores are normalized across all experts, the sparse and dense forms must agree.

In [ ]:
a = model.forward(x, top_k=8)
b = model.dense_reference(x)
print("max absolute difference:", float(np.max(np.abs(a-b))))
assert np.allclose(a,b,rtol=1e-5,atol=1e-5)


## 4. Why theoretical arithmetic and measured time can disagree

In [ ]:
from moe_lab.benchmark import run
rows=run(n_experts=16, hidden=64, intermediate=128, tokens=64, top_ks=[1,2,4,8,16], repeats=2)
for row in rows: print(row["top_k"], row["estimated_expert_and_router_MACs_per_token"], row["elapsed_ms_CPU_median"])


## 5. Hardware feasibility (theoretical lower bounds)

The model figures are **architecture-level** and do not represent successful GPU launches.

In [ ]:
from moe_lab.planner import MODELS, estimate
for name, (total, active) in MODELS.items():
    cap=estimate(total, bits=4, gpu_gb=80, gpus_per_node=8)
    print(name, f"{cap.raw_weight_gb:.0f} GB", cap.min_gpus, "80GB GPUs", cap.min_nodes, "nodes")


## 6. Challenge

Add an expert load-balancing penalty or compare conventional KV caches for different head counts. Explain why neither of these simplified estimates captures multi-node expert dispatch or modern MLA.